# Optimizing Churn Prediction: From Model Selection to Hyperparameter Tuning


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [3]:
# Load data
df = pd.read_csv('telecom_churn.csv')

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split data into train/test sets using an 80/20 split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify categorical and numeric columns
categorical_cols = X_train.select_dtypes(include=['object']).columns
numeric_cols = X_train.select_dtypes(exclude=['object']).columns

# Preprocessing step: apply StandardScaler to numeric columns and OneHotEncoder to categorical columns
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

#### 1. How many customers churned?

In [5]:
# Enter your answer in the left panel
churn_count = df['Churn'].value_counts()
churn_count

Churn
0    5174
1    1869
Name: count, dtype: int64

#### 2. Train a baseline model and evaluate with single split

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Use the already defined 'preprocess' pipeline
baseline_model = Pipeline([
    ('preprocess', preprocess),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

# Fit the model
baseline_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                ('logreg', LogisticRegression(max_iter=1000, random_state=42))])

#### 3. What is the single split accuracy?

In [20]:
# Enter your answer in the left panel
accuracy = baseline_model.score(X_test, y_test)
round(accuracy, 4)

0.8261

#### 4. Implement K-Fold Cross-Validation

In [23]:
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation
cv_scores = cross_val_score(baseline_model, X, y, cv=5, scoring='accuracy')

# Calculate mean CV score
mean_cv_score = cv_scores.mean()

print(f"CV scores: {cv_scores}")
print(f"Mean CV accuracy: {mean_cv_score:.4f} (+/- {cv_scores.std():.4f})")

CV scores: [0.80411639 0.81476224 0.78779276 0.80681818 0.80255682]
Mean CV accuracy: 0.8032 (+/- 0.0088)


#### 5. Which of these are hyperparameters?

In [ ]:
# select the correct options in the left panel

#### 6. Create a pipeline with preprocessing

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

#### 7. Create and fit GridSearchCV

In [29]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    # Regularization strength:
    # Smaller values -> stronger regularization (more penalty, simpler model)
    # Larger values -> weaker regularization (less penalty, more complex model)
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    
    # Type of regularization penalty:
    # 'l1' -> Lasso-style regularization (can drive coefficients to zero, feature selection)
    # 'l2' -> Ridge-style regularization (shrinks coefficients but keeps all features)
    'classifier__penalty': ['l1', 'l2'],

    # Optimization algorithm used to train the model:
    # 'liblinear' supports both L1 and L2 penalties
    # Suitable for small to medium-sized datasets and binary classification
    'classifier__solver': ['liblinear']
}

# Create GridSearchCV
grid_search = GridSearchCV(estimator=pipeline, param_grid= param_grid, cv=5, scoring='accuracy')

# Fit on training data
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

Best parameters: {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV score: 0.8010


#### 8. Create and fit RandomizedSearchCV

In [31]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# Define parameter distributions
param_distributions = {
    # Inverse regularization strength:
    # Smaller values -> stronger regularization (simpler model, less overfitting)
    # Larger values -> weaker regularization (more flexible model, risk of overfitting)
    # We sample C from a continuous range to let the model find the best balance
    'classifier__C': uniform(0.01, 100),
    
    # Type of regularization penalty:
    # 'l1' -> Lasso-style regularization (can drive coefficients to zero, feature selection)
    # 'l2' -> Ridge-style regularization (shrinks coefficients but keeps all features)
    'classifier__penalty': ['l1', 'l2'],

    # Optimization algorithm used to train the model:
    # 'liblinear' supports both L1 and L2 penalties
    # Suitable for small to medium-sized datasets and binary classification
    'classifier__solver': ['liblinear']
}

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distributions, n_iter=20, cv=5, scoring=
                                  'accuracy', random_state=42)

# Fit on training data
random_search.fit(X_train, y_train)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")

Best parameters: {'classifier__C': np.float64(61.17531604882809), 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
Best CV score: 0.8007


#### 9. when would you prefer RandomizedSearchCV

In [ ]:
# select the correct option in the left panel

#### 10. Evaluate the tuned model on test set

In [35]:
# Evaluate best model on test set
final_test_score = grid_search.best_estimator_.score(X_test, y_test)

# Print comparison
print("="*50)
print("PERFORMANCE COMPARISON")
print("="*50)
print(f"Single split baseline:  {accuracy:.4f}")
print(f"Best CV score (tuned):  {grid_search.best_score_:.4f}")
print(f"Final test score:       {final_test_score:.4f}")
print("="*50)

PERFORMANCE COMPARISON
Single split baseline:  0.8261
Best CV score (tuned):  0.8010
Final test score:       0.8226


#### 11. What did we accomplish?

In [ ]:
# select the correct options in the left panel